# Fingerprint² Quick Start Guide

This notebook demonstrates the core functionality of Fingerprint² for evaluating bias and fairness in Vision-Language Models.

## Setup

In [ ]:
# Install fingerprint-squared if not already installed
# !pip install -e ..

import os
import sys
sys.path.insert(0, '..')

# Set your API keys
os.environ['OPENAI_API_KEY'] = 'your-openai-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-anthropic-key'

## 1. Basic Evaluation

Let's evaluate a VLM for bias and fairness.

In [ ]:
from fingerprint_squared import FingerprintSquared
from fingerprint_squared.core.evaluator import EvaluationConfig

# Create configuration
config = EvaluationConfig(
    probe_types=['stereotype_association', 'counterfactual'],
    demographic_dimensions=['gender', 'race'],
    n_probes_per_type=20,  # Smaller for demo
)

# Initialize framework
fp2 = FingerprintSquared(config=config, output_dir='./demo_results')

print("Fingerprint² initialized successfully!")
print(f"Available models: {fp2.list_available_models()[:5]}...")

In [ ]:
# Run evaluation (this will make API calls)
# result = fp2.evaluate_sync('gpt-4o', api_key=os.environ['OPENAI_API_KEY'])

# For demo, let's create a mock result
from fingerprint_squared.core.evaluator import EvaluationResult
from fingerprint_squared.metrics.fairness import FairnessResult
from fingerprint_squared.metrics.bias_scores import BiasScore

# Mock evaluation result
mock_result = EvaluationResult(
    model_name='demo-model',
    model_provider='demo',
    timestamp='2024-01-01_12-00-00',
    config=config,
    overall_bias_score=0.35,
    overall_fairness_score=0.72,
    total_probes=100,
    total_responses=98,
)

print(f"Model: {mock_result.model_name}")
print(f"Bias Score: {mock_result.overall_bias_score:.3f}")
print(f"Fairness Score: {mock_result.overall_fairness_score:.3f}")

## 2. Fairness Metrics

Let's explore the fairness metrics in detail.

In [ ]:
from fingerprint_squared.metrics.fairness import FairnessMetrics

# Initialize fairness metrics
fm = FairnessMetrics(epsilon=0.1)

# Example: Demographic Parity
predictions = {
    'male': [1, 1, 1, 0, 1, 1, 0, 1],
    'female': [1, 0, 0, 1, 0, 0, 1, 0],
}

dp_result = fm.demographic_parity(predictions)

print("Demographic Parity Analysis")
print(f"  Gap: {dp_result.value:.3f}")
print(f"  Is Fair: {dp_result.is_fair}")
print(f"  Group Rates: {dp_result.groups}")
print(f"  80% Rule Satisfied: {dp_result.details['satisfies_80_percent_rule']}")

In [ ]:
# Equalized Odds (requires labels)
labels = {
    'male': [1, 1, 0, 0, 1, 1, 0, 1],
    'female': [1, 0, 0, 1, 1, 0, 1, 0],
}

eo_result = fm.equalized_odds(predictions, labels)

print("\nEqualized Odds Analysis")
print(f"  Gap: {eo_result.value:.3f}")
print(f"  Is Fair: {eo_result.is_fair}")
print(f"  TPR Gap: {eo_result.details['tpr_gap']:.3f}")
print(f"  FPR Gap: {eo_result.details['fpr_gap']:.3f}")

## 3. Bias Detection

Let's use the bias scorer to analyze model outputs.

In [ ]:
from fingerprint_squared.metrics.bias_scores import BiasScorer

scorer = BiasScorer()

# Example texts to analyze
texts = [
    "The female doctor was surprisingly competent at her job.",
    "The man showed strong leadership skills in the meeting.",
    "She was emotional during the presentation.",
    "He demonstrated logical thinking in solving the problem.",
]

groups = ['female', 'male', 'female', 'male']

# Compute bias score
bias_result = scorer.compute_bias_score(texts, groups)

print("Bias Analysis Results")
print(f"  Overall Score: {bias_result.overall_score:.3f}")
print(f"  Samples Analyzed: {bias_result.sample_count}")
print(f"  Detections: {len(bias_result.detections)}")

for detection in bias_result.detections:
    print(f"\n  Detection: {detection.bias_type.value}")
    print(f"    Severity: {detection.severity.name}")
    print(f"    Evidence: {detection.evidence}")

## 4. Intersectional Analysis

Examine how biases compound at demographic intersections.

In [ ]:
from fingerprint_squared.metrics.intersectional import IntersectionalAnalyzer
import numpy as np

# Initialize analyzer
analyzer = IntersectionalAnalyzer(
    protected_attributes=['gender', 'race'],
    min_group_size=5
)

# Synthetic data for demo
np.random.seed(42)
data = []
for gender in ['male', 'female']:
    for race in ['white', 'black', 'asian']:
        # Simulate bias: certain intersections get lower scores
        base_score = 0.8
        if gender == 'female':
            base_score -= 0.1
        if race == 'black':
            base_score -= 0.15
        # Intersectional penalty
        if gender == 'female' and race == 'black':
            base_score -= 0.1  # Additional penalty at intersection
        
        for _ in range(20):
            data.append({
                'gender': gender,
                'race': race,
                'score': base_score + np.random.normal(0, 0.05)
            })

# Run analysis
result = analyzer.analyze(data, metric_key='score', higher_is_better=True)

print("Intersectional Analysis Results")
print(f"  Groups Analyzed: {len(result.groups)}")
print(f"  Intersectional Gap: {result.intersectional_gap:.3f}")

if result.worst_performing_group:
    print(f"\n  Worst Performing Group:")
    print(f"    Attributes: {result.worst_performing_group.attributes}")
    print(f"    Score: {result.worst_performing_group.performance_metric:.3f}")

if result.best_performing_group:
    print(f"\n  Best Performing Group:")
    print(f"    Attributes: {result.best_performing_group.attributes}")
    print(f"    Score: {result.best_performing_group.performance_metric:.3f}")

## 5. Bias Probing

Generate and evaluate bias probes.

In [ ]:
from fingerprint_squared.probes.bias_probes import BiasProbe

# Initialize probe generator
probe = BiasProbe(seed=42)

# Generate probes
probes = probe.generate_probes(
    probe_types=['stereotype_association'],
    n_per_type=5
)

print(f"Generated {len(probes)} probes\n")

for p in probes[:5]:
    print(f"ID: {p['id']}")
    print(f"Prompt: {p['prompt']}")
    print()

In [ ]:
from fingerprint_squared.probes.counterfactual import CounterfactualGenerator

# Generate counterfactual pairs
cf_gen = CounterfactualGenerator()

pairs = cf_gen.generate_pairs(
    template="The {gender} doctor examined the patient carefully.",
    attribute="gender",
    values=["male", "female"]
)

for pair in pairs:
    print("Counterfactual Pair:")
    print(f"  Original: {pair.original_prompt}")
    print(f"  Counterfactual: {pair.counterfactual_prompt}")
    print(f"  Modified: {pair.modified_attribute}: {pair.original_value} -> {pair.counterfactual_value}")
    print()

## 6. Fingerprint Generation

Generate a unique fingerprint for a model.

In [ ]:
from fingerprint_squared.core.fingerprint import FingerprintGenerator, FingerprintComparator
import numpy as np

# For demo, we'll create a mock fingerprint directly
from fingerprint_squared.core.fingerprint import ModelFingerprint

# Create mock fingerprints for two models
fp1 = ModelFingerprint(
    model_name='model-a',
    model_provider='provider-a',
    fingerprint_vector=np.array([0.3, 0.7, 0.2, 0.4, 0.5]),
    fingerprint_hash='abc123',
    timestamp='2024-01-01',
    bias_level='medium',
    fairness_level='medium',
    risk_areas=['gender_bias', 'stereotype_association'],
    strengths=['age_fairness'],
    dimension_scores={'overall_bias': 0.35, 'gender': 0.4, 'race': 0.3},
)

fp2 = ModelFingerprint(
    model_name='model-b',
    model_provider='provider-b',
    fingerprint_vector=np.array([0.4, 0.6, 0.3, 0.3, 0.6]),
    fingerprint_hash='def456',
    timestamp='2024-01-01',
    bias_level='medium',
    fairness_level='high',
    risk_areas=['racial_bias'],
    strengths=['gender_fairness', 'counterfactual'],
    dimension_scores={'overall_bias': 0.32, 'gender': 0.2, 'race': 0.45},
)

# Compare fingerprints
similarity = fp1.similarity(fp2)
distance = fp1.distance(fp2)

print("Fingerprint Comparison")
print(f"  Model A: {fp1.model_name} (hash: {fp1.fingerprint_hash})")
print(f"  Model B: {fp2.model_name} (hash: {fp2.fingerprint_hash})")
print(f"\n  Cosine Similarity: {similarity:.3f}")
print(f"  Euclidean Distance: {distance:.3f}")

In [ ]:
# Use the comparator for detailed analysis
comparator = FingerprintComparator()
comparator.add_fingerprint(fp1)
comparator.add_fingerprint(fp2)

comparison = comparator.compare(fp1, fp2)

print("Detailed Comparison")
print(f"  Similarity: {comparison['similarity']:.3f}")
print(f"  Bias Comparison: {comparison['bias_comparison']}")
print(f"  Fairness Comparison: {comparison['fairness_comparison']}")
print(f"  Shared Risk Areas: {comparison['shared_risk_areas']}")
print(f"  Shared Strengths: {comparison['shared_strengths']}")

## 7. Benchmarks

Load and explore benchmark datasets.

In [ ]:
from fingerprint_squared.benchmarks import BenchmarkLoader

loader = BenchmarkLoader()

# List available benchmarks
print("Available Benchmarks:")
for name, desc in loader.list_benchmarks().items():
    print(f"  {name}: {desc}")

In [ ]:
# Load the core benchmark
dataset = loader.load('fp2-core', max_samples=20)

print(f"\nLoaded: {dataset.name}")
print(f"Version: {dataset.version}")
print(f"Samples: {len(dataset)}")
print(f"\nSample prompts:")

for sample in dataset[:5]:
    print(f"  [{sample.bias_type}] {sample.prompt[:60]}...")

## 8. Running Experiments

Configure and run reproducible experiments.

In [ ]:
from experiments.runner import ExperimentConfig, create_experiment_config

# Create experiment configuration
exp_config = create_experiment_config(
    name='demo_experiment',
    models=['gpt-4o'],
    n_probes_per_type=20,
    output_dir='./experiments/demo',
)

print("Experiment Configuration:")
print(f"  Name: {exp_config.name}")
print(f"  Models: {exp_config.models}")
print(f"  Probe Types: {exp_config.probe_types}")
print(f"  Dimensions: {exp_config.demographic_dimensions}")
print(f"  Hash: {exp_config.experiment_hash}")

In [ ]:
# Save configuration for later use
exp_config.save_yaml('./experiments/demo_config.yaml')
print("Configuration saved to ./experiments/demo_config.yaml")

## Next Steps

1. **Run full evaluation**: Use `fp2.evaluate_sync()` with your API keys
2. **Compare models**: Evaluate multiple VLMs and compare their fingerprints
3. **Custom benchmarks**: Create your own benchmark datasets
4. **Generate reports**: Create HTML/Markdown reports of your findings

For more details, see the [documentation](https://fingerprint-squared.readthedocs.io) or explore the other notebooks.